Required Imports

In [ ]:

!pip install torch transformers accelerate  tenacity

!!pip install torch --extra-index-url https://download.pytorch.org/whl/cu118


!pip install sentence-transformers
!pip install evaluate rouge_score nltk

In [ ]:
!pip uninstall torch torchvision torchaudio transformers -y
!pip install --upgrade pip setuptools wheel

Found existing installation: torch 2.0.1+cu118
Uninstalling torch-2.0.1+cu118:
  Successfully uninstalled torch-2.0.1+cu118
Found existing installation: torchvision 0.15.2+cu118
Uninstalling torchvision-0.15.2+cu118:
  Successfully uninstalled torchvision-0.15.2+cu118
Found existing installation: torchaudio 2.0.2+cu118
Uninstalling torchaudio-2.0.2+cu118:
  Successfully uninstalled torchaudio-2.0.2+cu118
Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-2

Finetune Lora

In [ ]:
import os
import time
import json
import torch
import sys
from packaging import version
from huggingface_hub import login


try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        AutoConfig,
        BitsAndBytesConfig,
        TrainingArguments,
        EarlyStoppingCallback,
    )
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
    from trl import SFTTrainer
    from datasets import Dataset
    from sklearn.model_selection import train_test_split
    import accelerate
    import transformers
    import peft
except ImportError as e:
    print(" Critical imports failed! Please install:")
    print("!pip install transformers==4.41.0 peft==0.10.0 accelerate==0.27.2 bitsandbytes==0.42.0 trl==0.8.6 scikit-learn")
    raise

# Verify versions
required_versions = {
    'transformers': '4.41.0',
    'peft': '0.10.0',
    'accelerate': '0.27.2'
}

for pkg, ver in required_versions.items():
    current_version = globals()[pkg].__version__
    if version.parse(current_version) != version.parse(ver):
        print(f" Version mismatch: {pkg}=={current_version} (required {ver})")
        sys.exit(1)

# Check GPU compatibility
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This script requires GPU acceleration.")
if torch.cuda.get_device_capability()[0] < 8:  # Ampere architecture check
    print(" Warning: Your GPU may not fully support bfloat16 operations")


MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DATASET_PATH = "unani_train_en_ur.json"
OUTPUT_DIR = f"unani-lora-ft-{int(time.time())}"
HF_TOKEN = ""  # Replace with your token

# Login to Hugging Face
login(token=HF_TOKEN)

os.makedirs(OUTPUT_DIR, exist_ok=True)


def format_instruction(sample):
    system_prompt = """You are an expert in Unani medicine. Provide accurate, detailed information about Unani treatments, principles, and remedies based on the user's questions."""

    # Handle both single-hop and multi-hop questions
    instruction = sample.get('instruction', '')
    output = sample.get('output', '')

    # Include Urdu translations if available
    ur_instruction = sample.get('instruction_ur', '')
    ur_output = sample.get('output_ur', '')

    full_instruction = f"{instruction}\n{ur_instruction}" if ur_instruction else instruction
    full_output = f"{output}\n{ur_output}" if ur_output else output

    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{full_instruction}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{full_output}<|eot_id|>"""

def load_and_process_data(file_path, test_size=0.1):
    with open(file_path) as f:
        data = json.load(f)

    
    if not isinstance(data, list):
        raise ValueError("Dataset should be a list of samples")

    valid_samples = []
    for sample in data:
        if isinstance(sample, dict) and 'instruction' in sample and 'output' in sample:
            valid_samples.append(sample)

    if not valid_samples:
        raise ValueError("No valid samples found in the dataset")

    formatted_data = [{"text": format_instruction(sample)} for sample in valid_samples]
    train_data, val_data = train_test_split(formatted_data, test_size=test_size, random_state=42)
    return train_data, val_data

def load_model():
    # Simplified loading approach that works with Llama 3
    try:
        print(" Attempting to load model with 4-bit quantization")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            token=HF_TOKEN,
            torch_dtype=torch.float16
        )
        print(" Model loaded successfully with 4-bit quantization")
        return model
    except Exception as e:
        print(f" 4-bit loading failed: {str(e)}")
        torch.cuda.empty_cache()

    try:
        print(" Attempting to load model with 8-bit quantization")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            load_in_8bit=True,
            device_map="auto",
            token=HF_TOKEN,
            torch_dtype=torch.float16
        )
        print(" Model loaded successfully with 8-bit quantization")
        return model
    except Exception as e:
        print(f" 8-bit loading failed: {str(e)}")
        torch.cuda.empty_cache()

    try:
        print(" Attempting to load model without quantization")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            token=HF_TOKEN,
            torch_dtype=torch.float16
        )
        print(" Model loaded successfully without quantization")
        return model
    except Exception as e:
        print(f" Standard loading failed: {str(e)}")
        torch.cuda.empty_cache()

    raise RuntimeError("All model loading strategies failed")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    padding_side="right",
    use_fast=False,
    truncation_side="left"
)
tokenizer.pad_token = tokenizer.eos_token

model = load_model()


try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
except Exception as e:
    print(f" Couldn't prepare for k-bit training: {str(e)}")
    print("Proceeding without k-bit optimization")

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)

# Data preparation
train_data, val_data = load_and_process_data(DATASET_PATH)
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)


mixed_precision = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
print(f" Using {mixed_precision} precision")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=(mixed_precision == "fp16"),
    bf16=(mixed_precision == "bf16"),
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    warmup_ratio=0.1,
    max_grad_norm=0.3,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    load_best_model_at_end=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


try:
    print(" Starting training...")
    trainer.train()

    # Save outputs
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f" Training complete! Model saved to: {OUTPUT_DIR}")

except Exception as e:
    print(f" Training failed: {str(e)}")
    raise
finally:
    torch.cuda.empty_cache()

⚠️ Warning: Your GPU may not fully support bfloat16 operations


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


🔄 Attempting to load model with 4-bit quantization


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully with 4-bit quantization
ℹ️ Using fp16 precision


...

🚀 Starting training...


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
50,0.861000,0.931854
100,0.536100,0.647308
150,0.553500,0.557643
200,0.516000,0.541295


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in t

✅ Training complete! Model saved to: unani-lora-ft-1753643729
